# Lab 21 — Fine-tuning LLMs · RUN ALL (T4)

Chay tu tren xuong. Runtime > Change runtime type > **T4 GPU** truoc khi bat dau.

| O | Lam gi | Thoi gian |
|---|---|---|
| 1 | clone + install | ~1 phut |
| 2 | smoke: import + unit test | ~30 giay |
| 3 | **core pipeline NB1 -> NB5** | ~80 phut |
| 4 | gatekeeper + in ket qua | ~10 giay |


In [1]:
# @title 1. Setup — clone + install (chạy ô này trước)
import os, subprocess, sys

REPO = "https://github.com/ngnkhanhly7/K3-DAY21-2A202601403-NguyenThiKhanhLy.git"
if not os.path.exists("K3-DAY21-2A202601403-NguyenThiKhanhLy"):
    subprocess.run(["git", "clone", "-q", REPO], check=True)
os.chdir("/content/K3-DAY21-2A202601403-NguyenThiKhanhLy")
subprocess.run(["git", "pull", "-q"], check=False)
sys.path.insert(0, "src")

# Install from requirements.txt, NOT a copied list. The copied list is how the
# torchao>=0.16 pin reached requirements.txt and this bootstrap on different days --
# and a bootstrap missing a pin does not fail here, it fails 10 minutes later inside
# get_peft_model(). One source of truth. torch is preinstalled on Colab and
# requirements.txt pins it compatibly, so that line is a no-op.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
               check=True)

import torch
print("commit :", subprocess.run(["git","rev-parse","--short","HEAD"],
                                 capture_output=True, text=True).stdout.strip())
print("GPU    :", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "NONE — Runtime > Change runtime type > T4 GPU")
if torch.cuda.is_available():
    print("VRAM   : %.1f GB" % (torch.cuda.get_device_properties(0).total_memory/1024**3))


commit : 11fedf3
GPU    : NVIDIA L4
VRAM   : 22.0 GB


In [2]:
# @title 2. Smoke — imports, seed data, unit tests (no GPU needed)
!python scripts/verify.py --smoke



[  ok  ] labkit imports                                   
[  ok  ] tier resolves                                    T4 -> unsloth/Qwen3.5-4B
[  ok  ] all tiers respect the <32 effective-batch rule   
[  ok  ] data/train_seed.jsonl                            250 rows
[  ok  ] data/eval_target.jsonl                           50 rows
[  ok  ] data/eval_regression.jsonl                       15 rows
[ FAIL ] unit tests                                       3 failed, 116 passed in 2.15s

6 passed · 0 warnings · 1 failures

Not ready to submit — fix the FAILs above.


In [3]:
# @title 3. Core pipeline — NB1 → NB5
# EVAL_LIMIT truncates both eval sets: "" = full run (submittable),
# 8 = ~fast smoke pass. STAGES lets you resume after a failure.
import os
COMPUTE_TIER = "BIGGPU"        # @param ["CPU","LAPTOP","T4","BIGGPU"]
EVAL_LIMIT   = ""         # @param ["", "4", "8", "16", "25"]
STAGES       = "nb1 nb2 nb3 nb4 nb5"   # @param {type:"string"}

os.environ["COMPUTE_TIER"] = COMPUTE_TIER
if EVAL_LIMIT:
    os.environ["EVAL_LIMIT"] = EVAL_LIMIT
else:
    os.environ.pop("EVAL_LIMIT", None)

from labkit import device
print(device.banner(), "\n")

!python scripts/colab_run.py {STAGES}


NVIDIA L4 (cuda, sm_89, 22.0 GB) -> precision=bf16 

tier=BIGGPU  mask=assistant-only  eval_limit=full

NB1 — data, chat template & loss mask
tier=BIGGPU  model=Qwen/Qwen3.5-9B  max_length=192
250 mẫu huấn luyện
{
  "instruction": "Phân loại ticket chăm sóc khách hàng sau thành JSON với đúng 4 khóa: intent, urgency, product, sentiment. Chỉ trả về JSON, không giải thích.\n\nintent thuộc: doi_tra | van_chuyen | hoan_tien | san_pham_loi | hoi_thong_tin\nurgency thuộc: cao | trung_binh | thap\nsentiment thuộc: tieu_cuc | trung_tinh | tich_cuc\nproduct: tên sản phẩm xuất hiện trong ticket.",
  "input": "Alo sh
eos_token: <|im_end|>
VERDICT: reasoning preserved — safe to train on traces

--- chuỗi đã render ---
<|im_start|>user
2+2?<|im_end|>
<|im_start|>assistant
<think>
buoc 1: kiem tra. buoc 2: tra loi.
</think>

4<|im_end|>

mode = assistant-only   supervised 39/94 (41%)
--- LOSS TÍNH TRÊN ĐOẠN NÀY ---
</think>

{"intent": "doi_tra", "urgency": "trung_binh", "product": "balo laptop", "se

In [5]:
%cd /content/K3-DAY21-2A202601403-NguyenThiKhanhLy
!git fetch origin main
!git reset --hard origin/main
!git rev-parse --short HEAD

/content
remote: Enumerating objects: 1, done.
remote: Counting objects: 100% (1/1), done.
remote: Total 1 (delta 0), reused 1 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (1/1), 183 bytes | 183.00 KiB/s, done.
From https://github.com/ngnkhanhly7/K3-DAY21-2A202601403-NguyenThiKhanhLy
 * branch            main       -> FETCH_HEAD
   11fedf3..eb5b88c  main       -> origin/main
HEAD is now at eb5b88c Restore generated run-all notebook
eb5b88c


In [7]:
# @title 4. Gatekeeper + results
!python scripts/verify.py
print("\n================ results/ ================")
!ls -la results/
!echo && echo "---- runs.csv ----" && cat results/runs.csv 2>/dev/null
!echo && echo "---- verdict.json ----" && cat results/verdict.json 2>/dev/null



[  ok  ] labkit imports                                   
[  ok  ] tier resolves                                    BIGGPU -> Qwen/Qwen3.5-9B
[  ok  ] all tiers respect the <32 effective-batch rule   
[  ok  ] data/train_seed.jsonl                            250 rows
[  ok  ] data/eval_target.jsonl                           50 rows
[  ok  ] data/eval_regression.jsonl                       15 rows
[ FAIL ] unit tests                                       3 failed, 116 passed in 1.97s
[  ok  ] results/template_check.json                      
[  ok  ] results/mask_proof.json                          
[  ok  ] results/token_stats.json                         
[  ok  ] results/baselines_frozen.json                    
[  ok  ] results/runs.csv                                 
[  ok  ] results/verdict.json                             
[  ok  ] results/autopsy.json                             
[  ok  ] submission/REPORT.md                             
[ FAIL ] REPORT.md filled in          

In [9]:
%cd /content/K3-DAY21-2A202601403-NguyenThiKhanhLy
!git status --short

/content
?? K3-DAY21-2A202601403-NguyenThiKhanhLy/


In [5]:
from labkit.config import get_tier, SPECS
from labkit import train

tier = get_tier("BIGGPU")
print(tier)
print(train.sft_config_kwargs(tier, SPECS["correct"], "x")["loss_type"])

Tier(name='BIGGPU', model_id='Qwen/Qwen3.5-9B', vram_gb_bf16_lora=22.0, max_length=192, per_device_batch=1, grad_accum=16, notes='L4 22.5 GB / A100 40 GB / RTX 3090-4090.')
nll


In [6]:
!rm -rf adapters/correct
!python scripts/colab_run.py nb3

tier=BIGGPU  mask=assistant-only  eval_limit=full

NB3 — train the correct configuration
BIGGPU · Qwen/Qwen3.5-9B · all-linear · r=16 · LR 10x · 16-bit
NVIDIA L4 (cuda, sm_89, 22.0 GB) -> precision=bf16
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files: 100% 4/4 [00:00<00:00, 1484.32it/s]
Download complete: :           |  0.00B            
Download complete: :           |  0.00B            
Reconstruction complete: |          |  0.00B /  0.00B            
Loading weights: 100% 427/427 [00:04<00:00, 95.32it/s] 
{
  "num_hidden_layers": 32